# Lung atlas (GSE130148) multi-cluster comparison

Runs all 6 methods on the 4 leave-one-Dropseq-batch-out comparisons (each batch as
target, the other three as sources), recovering all K=13 cell types. Metrics: our own
misclustering error (Hungarian-matched), ARI, V-measure.

Each method lives in its own script (`method_ours.py` for our 4 existing methods,
`method_tlgmm.py`, `method_scrna.py`, `method_gdec.py`), imported and called
**one at a time** below with its own timing, so a slow method or a failure is easy to
isolate. Shared data loading/metrics live in `common.py`. The LaTeX table generator is
defined directly in this notebook (it's presentation logic, not part of any method).

**Environment**: `transfer_clustering` (conda). Before running `scrna` / `gdec_gcnfree`
you'll need their extra dependencies, not installed there by default:
```
conda activate transfer_clustering
pip install cvxopt   # scRNA's NMF pipeline
pip install torch     # GDEC's SDAE/DEC pipeline (CPU-only is fine at this data scale)
```

In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd

import common

In [2]:
H5AD_PATH = os.path.join("..", "lung_atlas_analysis", "data", "lung_atlas_hvg_lognorm.h5ad")

batch_data = common.load_batches(H5AD_PATH)
for b, (X, y) in batch_data.items():
    print(f"{b}: {X.shape[0]} cells x {X.shape[1]} genes, {len(set(y))} cell types")

Dropseq_1: 2098 cells x 5000 genes, 13 cell types
Dropseq_2: 3183 cells x 5000 genes, 13 cell types
Dropseq_3: 2387 cells x 5000 genes, 13 cell types
Dropseq_4: 2273 cells x 5000 genes, 13 cell types


In [ ]:
TARGET_BATCH = "Dropseq_1"   # edit and re-run the cells below to sanity-check timing
                              # on a single batch before running the full loop

X_T, y_T = batch_data[TARGET_BATCH]
sources = [batch_data[b][0] for b in common.BATCHES if b != TARGET_BATCH]
print(f"target={TARGET_BATCH}, n_T={X_T.shape[0]}, sources n=({', '.join(str(s.shape[0]) for s in sources)})")

target=Dropseq_1, n_T=2098, sources n=(3183, 2387, 2273)


## Run each method one at a time (timed)

Each cell below imports and runs exactly one method script. If one is slow or errors
out, the others are unaffected -- rerun just that cell after adjusting its script's
constants (e.g. `method_tlgmm.MTL_EM_ITERS`, `method_gdec.GDEC_PRETRAIN_EPOCHS`).

In [ ]:
def timed_run(name, fn, X_T, sources, seed, y_T, timings, rows):
    t0 = time.time()
    z_hat = fn(X_T, sources, seed)
    elapsed = time.time() - t0
    timings[name] = elapsed
    metrics = common.compute_metrics(z_hat, y_T)
    rows.append(dict(target=TARGET_BATCH, method=name, elapsed_sec=elapsed, **metrics))
    print(f"{name}: {elapsed:.1f}s -- {metrics}")
    return z_hat

timings = {}
rows = []

In [ ]:
import method_ours

for name, fn in method_ours.OURS_METHODS.items():
    timed_run(name, fn, X_T, sources, common.RANDOM_STATE, y_T, timings, rows)

target_only: 64.3s -- {'misclustering': 0.4518589132507149, 'ari': 0.4415398894353676, 'v_measure': 0.6503306644222021}
multi_source_pooled: 523.0s -- {'misclustering': 0.5319351763584366, 'ari': 0.3857767070199255, 'v_measure': 0.6458809566714967}


In [ ]:
import method_tlgmm

timed_run("tlgmm", method_tlgmm.method_tlgmm, X_T, sources, common.RANDOM_STATE, y_T, timings, rows)

In [ ]:
import method_scrna

timed_run("scrna", method_scrna.method_scrna, X_T, sources, common.RANDOM_STATE, y_T, timings, rows)

In [ ]:
import method_gdec

timed_run("gdec_gcnfree", method_gdec.method_gdec_gcnfree, X_T, sources, common.RANDOM_STATE, y_T, timings, rows)

In [ ]:
pd.DataFrame(rows)

## Aggregate cluster results

The full 4-batch x 6-method comparison now runs on the cluster as 24 independent
Slurm array tasks (`run_lung_atlas_comparison.py` / `run_lung_atlas_comparison.sh`),
each writing one row to `results/raw/<batch>__<method>.csv`. This cell just reads
those back in and concatenates them into the same `results` DataFrame the cells
below expect -- rerun it (after the array job finishes, or after rerunning any
individual task) to refresh the tables/LaTeX below.

In [ ]:
import glob

RAW_RESULTS_DIR = os.path.join("results", "raw")
raw_paths = sorted(glob.glob(os.path.join(RAW_RESULTS_DIR, "*.csv")))

expected = {(batch, method) for batch in common.BATCHES for method in common.METHOD_LABELS}
found = set()
all_rows = []
for path in raw_paths:
    row = pd.read_csv(path).iloc[0]
    all_rows.append(row.to_dict())
    found.add((row["target"], row["method"]))

missing = expected - found
if missing:
    print(f"WARNING: {len(missing)}/{len(expected)} (batch, method) results not found in {RAW_RESULTS_DIR}:")
    for batch, method in sorted(missing):
        print(f"  {batch}, {method}")

results = pd.DataFrame(all_rows)
results.to_csv(os.path.join("results", "lung_atlas_comparison_results.csv"), index=False)
results

## Timing summary (bottleneck check)

In [ ]:
results.pivot(index="method", columns="target", values="elapsed_sec").reindex(
    index=list(common.METHOD_LABELS.keys()), columns=common.BATCHES
)

## Summary tables (method x target batch)

In [ ]:
for metric in ["misclustering", "ari", "v_measure"]:
    print(f"\n--- {metric} ---")
    display(results.pivot(index="method", columns="target", values=metric)
                    .reindex(index=list(common.METHOD_LABELS.keys()), columns=common.BATCHES))

## LaTeX booktabs tables (best performer per column bolded)

This is presentation logic, so it lives here rather than in any method/common script.

In [ ]:
def make_latex_table(results: pd.DataFrame, metric: str, caption: str = "", label: str = "") -> str:
    """One booktabs table for a given metric: rows = methods, columns =
    target batches, best value per column bolded. `misclustering` is
    better when LOWER; ari/v_measure are better when HIGHER."""
    pivot = results.pivot(index="method", columns="target", values=metric)
    pivot = pivot.reindex(index=list(common.METHOD_LABELS.keys()), columns=common.BATCHES)
    better_low = metric == "misclustering"

    lines = []
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append("\\begin{tabular}{l" + "c" * len(common.BATCHES) + "}")
    lines.append("\\toprule")
    lines.append("Method & " + " & ".join(common.BATCHES).replace("_", "\\_") + " \\\\")
    lines.append("\\midrule")
    for method in pivot.index:
        cells = []
        for batch in common.BATCHES:
            val = pivot.loc[method, batch]
            best = pivot[batch].min() if better_low else pivot[batch].max()
            cell = f"{val:.3f}"
            if np.isclose(val, best):
                cell = f"\\textbf{{{cell}}}"
            cells.append(cell)
        label_str = common.METHOD_LABELS[method].replace("_", "\\_")
        lines.append(f"{label_str} & " + " & ".join(cells) + " \\\\")
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    if caption:
        lines.append(f"\\caption{{{caption}}}")
    if label:
        lines.append(f"\\label{{{label}}}")
    lines.append("\\end{table}")
    return "\n".join(lines)

In [ ]:
captions = {
    "misclustering": ("Misclustering error (lower is better) on the lung atlas, "
                       "leave-one-Dropseq-batch-out."),
    "ari": "Adjusted Rand Index (higher is better) on the lung atlas, leave-one-Dropseq-batch-out.",
    "v_measure": "V-measure (higher is better) on the lung atlas, leave-one-Dropseq-batch-out.",
}

latex_tables = {}
for metric, caption in captions.items():
    tex = make_latex_table(results, metric, caption=caption, label=f"tab:lung_{metric}")
    latex_tables[metric] = tex
    print(tex)
    print()

with open(os.path.join("results", "lung_atlas_tables.tex"), "w") as f:
    for metric, tex in latex_tables.items():
        f.write(f"% --- {metric} ---\n")
        f.write(tex)
        f.write("\n\n")
print("Wrote results/lung_atlas_tables.tex")